In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm
from torchinfo import summary
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import gzip
import pickle

from torch.autograd import Variable

torch.backends.cudnn.enabled = False
#random_seed = 42
#torch.manual_seed(random_seed)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

Here we download the datasets from Google Drive : (https://drive.google.com/drive/folders/1tgwAUkoPU1DwoUs_mTa9ZvGJ0cIhY3ob)

In [ ]:
!gdown 1BppDIfS3QqmnAz-ZZzoJ19rwFwetaH17
!gdown 1o5Xa6nmiNYgFuoGTYCRYXKKKUkV3bxJL
!gdown 1ojvAvkjDMruiY-q6Ev8LDDDPpOn1gcYT
!gdown 1ho4_rFWJ4MZMzyHdm_qlgStm2eGFuQDq
!gdown 1n8iIzBxaWeLRahQkKhs1dkuBfSm8INbN
!wget https://storage.googleapis.com/cvdf-datasets/mnist/t10k-labels-idx1-ubyte.gz
!gunzip t10k-labels-idx1-ubyte.gz

# DATA Part

Unzip the files :

In [ ]:
# Function to load data from a .gz file
def load_from_gz(filename):
    with gzip.open(filename, 'rb') as f:
        data = pickle.load(f)
    return data

# Load the data
loaded_images = load_from_gz('colored_mnist_images.gz')
loaded_targets = load_from_gz('colored_mnist_targets.gz')

Create the dataset :

In [ ]:
# Function to show an image
def imshow(img):
    npimg = img.numpy()
    plt.imshow(np.transpose(npimg, (1, 2, 0)))
    plt.show()

#Function to split training into training+validation
from torch.utils.data import Subset
from sklearn.model_selection import train_test_split
def train_val_dataset(dataset, val_split=0.25):
    train_idx, val_idx = train_test_split(list(range(len(dataset))), test_size=val_split)
    datasets = {}
    datasets['train'] = Subset(dataset, train_idx)
    datasets['val'] = Subset(dataset, val_idx)
    return datasets
  
#Define Dataset
class ColoredMNISTDataset(Dataset):
    def __init__(self, images, targets):
        self.images = images
        self.targets = targets

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, idx):
        image = self.images[idx]
        target = self.targets[idx]
        return torch.tensor(image), target

# Create the dataset and split it
val_percent = 0
train_dataset_FULL = ColoredMNISTDataset(loaded_images, loaded_targets)
if val_percent>0:    
    SplitedDATA = train_val_dataset(train_dataset_FULL, val_split=val_percent)
    idx_train = SplitedDATA['train'].indices
    idx_val = SplitedDATA['val'].indices
    train_images, train_targets = [list(np.array(loaded_images)[idx_train]), list(np.array(loaded_targets)[idx_train])]
    val_images, val_targets = [list(np.array(loaded_images)[idx_val]), list(np.array(loaded_targets)[idx_val])]

    train_dataset = ColoredMNISTDataset(train_images, train_targets)
    val_dataset = ColoredMNISTDataset(val_images, val_targets)
else:
    print("No validation dataset")
    train_dataset = train_dataset_FULL  ;  val_dataset = train_dataset_FULL

Preprocess the images : 

Image normalization is the process of transforming the values of an image to enhance its contrast, reduce variations in lighting, and improve its overall quality.

Normalization standardizes the appearance of images, making them more suitable for processing algorithms and improving the robustness of the recognition system to variations in input.

- Resize the images to a standard size suitable for processing, typically scaling them to a specific resolution
- Grayscaled image
- Then contrast enhancement techniques applied to improve the visibility of text regions and reduce variations in lighting (~binarization step)
- Next step is to adjust the brightness of the image if necessary to ensure consistent illumination <br><br>
- Rotate to increase training dataset ???

In [ ]:
def PreProcess(images, resolution_scale=28, contrast_factor=1e10, brightness_factor=1e10):
    #Resize all images if needed
    Pimages = transforms.Resize(resolution_scale)(images)
    
    #Grayscale them
    Pimages = transforms.Grayscale()(Pimages)
    
    #Contrast enhancement
    Pimages = transforms.functional.adjust_contrast(Pimages, contrast_factor=contrast_factor)
    
    #Brightness adjustment
    Pimages = transforms.functional.adjust_brightness(Pimages, brightness_factor=brightness_factor)
    
    #Binarize
    Pimages = Pimages > 0.1
    Pimages = Pimages.float()
    
    return Pimages


resolution_scale = 28 #!!!!!!!!!!!!!!!!!!!!! COMPRESS IT ???
print("Preprocessing training dataset")
for i in tqdm(range(len(train_dataset.images))):
    train_dataset.images[i] = np.array(PreProcess(torch.tensor(train_dataset.images[i]), resolution_scale=resolution_scale).float())

print("Preprocessing validation dataset")
for i in tqdm(range(len(val_dataset.images))):
    val_dataset.images[i] = np.array(PreProcess(torch.tensor(val_dataset.images[i]), resolution_scale=resolution_scale).float())


In [ ]:
# Create a DataLoader for the datasets
batch_size = int(5e3)
train_dataloader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_dataloader = torch.utils.data.DataLoader(val_dataset, batch_size=batch_size, shuffle=True)

In [ ]:
# Get some random training images
dataiter = iter(train_dataloader)
images, labels = next(dataiter)

# Show images
imshow(torchvision.utils.make_grid(images[0:64]))

# Model Part

The training loop : 
 - avoid overfitting by stopping if validation performances drop ?

In [ ]:
class CNN_MNIST(nn.Module):
    def __init__(self, input_size, num_classes):
        """
        init convolution and activation layers
        Args:
            input_size: (1,resolution_scale,resolution_scale)
            num_classes: 10
        """
        super(CNN_MNIST, self).__init__()
        
        self.layer1 = nn.Sequential(
            nn.Conv2d(input_size[0], 32, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(32))
        
        self.layer2 = nn.Sequential(
            nn.Conv2d(32, 32, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(32))
        
        self.layer3 = nn.Sequential(
            nn.Conv2d(32, 32, kernel_size=5, stride=2, padding=0),
            nn.ReLU(),
            nn.BatchNorm2d(32),
            nn.Dropout(p=0.4))
        
        self.layer4 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(64))
        
        self.layer5 = nn.Sequential(
            nn.Conv2d(64, 64, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(64))
        
        self.layer6 = nn.Sequential(
            nn.Conv2d(64, 64, kernel_size=5, stride=2, padding=0),
            nn.ReLU(),
            nn.BatchNorm2d(64),
            nn.Dropout(p=0.4))
        
        self.layer7 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=4),
            nn.ReLU(),
            nn.BatchNorm2d(128),
            nn.Flatten(),
            nn.Dropout(p=0.4))
        
        self.out = nn.Linear(128, num_classes)
    
    def forward(self, x):
        """
        forward function describes how input tensor is transformed to output tensor
        Args:
            x: (Nx1xresolution_scalexresolution_scale) tensor
        """
        
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.layer5(x)
        x = self.layer6(x)
        x = self.layer7(x)
        
        x = x.reshape(x.size(0), -1)
        x= self.out(x)
        return x

In [ ]:
cnn = CNN_MNIST(input_size=(1, resolution_scale, resolution_scale), num_classes=10)
cnn = cnn.to(device)

num_epochs = 20  ;  total_loss = 0  ;  total_acc = 0

#'batch_size': 64

# Train the model
cnn.train()
total_step = len(train_dataloader)

optimizer = torch.optim.Adam(cnn.parameters(), 1e-3)
scheduler = torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=0.95, total_iters=total_step)
criterion = torch.nn.CrossEntropyLoss()  # loss function

for epoch in range(num_epochs):
    for i, (images, labels) in enumerate(train_dataloader):
        
        # gives batch data, normalize x when iterate train_loader
        b_x = Variable(images)   # batch x
        b_y = Variable(labels)   # batch youtput = cnn(b_x)[0]     
        outputs = cnn(b_x.float())
        loss = criterion(outputs, labels.float().type(torch.LongTensor))

        # clear gradients for this training step   
        optimizer.zero_grad()

        # backpropagation, compute gradients 
        loss.backward()                # apply gradients             
        optimizer.step()                

        total_loss+=loss.item()
        total_acc+=torch.sum(torch.max(outputs,dim=1)[1]==labels).item()*1.0    

        print('Epoch [{}/{}], Step [{}/{}], Loss: {:.4f}' 
               .format(epoch + 1, num_epochs, i + 1, total_step, loss.item()))
    scheduler.step()


In [ ]:
torch.save(cnn.state_dict(), "CNN_model8_weights.pth")
the_model = CNN_MNIST(input_size=(1, resolution_scale, resolution_scale), num_classes=10)
the_model.load_state_dict(torch.load("CNN_model8_weights.pth", weights_only=True))
the_model = the_model.to(device)

# Evaluate model on Validate dataset

In [ ]:
total_step = len(val_dataloader)
TP = 0  ;  total_points = 0
for i, (images, labels) in enumerate(val_dataloader):
    b_x = Variable(images)   # batch x
    b_y = Variable(labels)   # batch youtput = cnn(b_x)[0]
    outputs = the_model(b_x.float())
    
    predicted_labels = outputs.max(1).indices.cpu().numpy()
    true_labels = b_y.cpu().numpy()
    
    TP += int(torch.sum(b_y == outputs.max(1).indices))
    total_points += len(b_y)
    print("Step ["+str(i+1)+"/"+str(total_step)+"] - Accuracy : "+str(TP/total_points))

# Predict test dataset

Create your testset :

In [ ]:
loaded_test_images = load_from_gz('colored_mnist_test_images.gz')


class TestMNISTDataset(Dataset):
    def __init__(self, images):
        self.images = images

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = self.images[idx]
        return torch.tensor(image)

# Create the new dataset
test_dataset = TestMNISTDataset(loaded_test_images)

#Preprocess
for i in tqdm(range(len(test_dataset.images))):
    test_dataset.images[i] = PreProcess(torch.tensor(test_dataset.images[i]), resolution_scale=resolution_scale).numpy()
test_dataloader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
import idx2numpy
file = 't10k-labels-idx1-ubyte'
test_labels = idx2numpy.convert_from_file(file)

In [ ]:
total_step = len(test_dataloader)
TP = 0  ;  total_points = 0
for i, images in enumerate(test_dataloader):
    b_x = Variable(images)   # batch x
    b_y = Variable(torch.tensor(test_labels[i*batch_size:(i+1)*batch_size]))   # batch youtput = cnn(b_x)[0]     
    outputs = the_model(b_x.float())
    
    predicted_labels = outputs.max(1).indices.cpu().numpy()
    true_labels = b_y.cpu().numpy()
    
    TP += int(torch.sum(b_y == outputs.max(1).indices))
    total_points += len(b_y)
    print("Step ["+str(i+1)+"/"+str(total_step)+"] - Accuracy : "+str(TP/total_points))

Create your predictions.npy file, you don't have to modify this cell :

In [ ]:
predictions = []

for i, images in enumerate(test_dataloader):
    b_x = Variable(images)   # batch x
    b_y = Variable(torch.tensor(test_labels[i*batch_size:(i+1)*batch_size]))   # batch youtput = cnn(b_x)[0]     
    outputs = the_model(b_x.float())
    
    predicted_labels = list(outputs.max(1).indices)
    predictions = predictions+predicted_labels
np.save('predictions.npy', predictions)